In [1]:
import numpy as np
import random
import math
import copy
from YenAlgorithm import YenAlgorithm

class Ant(object):
    def __init__(self):
        self.path = []
        self.fitness = float('inf')
        self.delta = 0


class ACO:

    def __init__(self, src, dst, N, Max, K_paths, p, a, b, p0, Q):
        self.switches = [i for i in range(1,61)]
        self.src= src
        self.dst = dst
        self.weight_map= self.GetWeightMap()
        self.adjacency = copy.deepcopy(self.weight_map)
        self.N = N
        self.Max = Max
        self.K_paths = K_paths
        self.p = p
        self.a = a
        self.b = b
        self.p0 = p0
        self.Q = Q
        self.colony = [Ant() for i in range(self.N)]
        self.condidates = []
        self.best = []
        self.pheromone = self.CreatePheromone()
        self.probability = self.CreateProbability()
    
    def CreatePheromone(self):
        pheromone = copy.deepcopy(self.adjacency)
        for sw_1 in self.adjacency.keys():
            for sw_2 in self.adjacency[sw_1].keys():
                pheromone[sw_1][sw_2] = 1
        return pheromone

    def CreateProbability(self):
        probability = copy.deepcopy(self.adjacency)
        for sw_1 in self.adjacency.keys():
            for sw_2 in self.adjacency[sw_1].keys():
                probability[sw_1][sw_2] = 0
        return probability
    
    def GetWeightMap(self):
        weight_map={}
        temp = 0
        with open('metric_data_60.txt') as f:
            for line in f:
                strt = line
                strt2 = strt.split(':')
                my_result = list(map(int, strt2[0].split(',')))
                if (temp!=my_result[0]):
                    weight_map[my_result[0]]={}
                weight_map[my_result[0]][my_result[1]] = int(strt2[1])
                temp = my_result[0]
        return weight_map

    def CreatePath(self):
        for i in range(self.N):
            path = []
            current_switch = self.src
            path.append(current_switch)
            while(current_switch!=self.dst):
                self.probability = self.CreateProbability()
                neighbor_switches = set(self.adjacency[current_switch].keys())-set(path)
                neighbor_switches = list(neighbor_switches)
                if(len(neighbor_switches)==0):
                    path.clear()
                    current_switch = self.src
                    path.append(current_switch)
                else:
                    current_switch = self.GetNextSwitch(neighbor_switches, current_switch)
                    path.append(current_switch)
            self.colony[i].path = copy.deepcopy(path)
            self.colony[i].fitness = self.Evaluate(self.colony[i].path)
            self.colony[i].delta = 1/self.colony[i].fitness

    def GetNextSwitch(self, neighbor_switches, current_switch):
        sum = 0
        for sw in neighbor_switches:
            x = self.pheromone[current_switch][sw]
            y = 1/(self.weight_map[current_switch][sw])
            z = pow(x,self.a)*pow(y,self.b)
            sum+=z
        prob = []
        for sw in neighbor_switches:
            x = self.pheromone[current_switch][sw]
            y = 1/(self.weight_map[current_switch][sw])
            z = pow(x,self.a)*pow(y,self.b)
            self.probability[current_switch][sw]=z/sum
            self.probability[sw][current_switch]=z/sum
            prob.append(z/sum)
        sw_max = 1
        prob_max = 0
        for sw in neighbor_switches:
            if(self.probability[current_switch][sw]>=prob_max):
                prob_max = self.probability[current_switch][sw]
                sw_max = sw
        p = np.random.rand()
        if(p <= self.p0):
            current_switch= sw_max
        else:
            sw = np.random.choice(neighbor_switches,p=prob)
            current_switch= sw
        return current_switch

    def Evaluate(self,path):
        calculatedFitness = 0
        for i in range(len(path) - 1):
            p1 = path[i]
            p2 = path[i + 1]
            calculatedFitness += self.weight_map[p1][p2]
        return calculatedFitness
    
    def UpdatePheromone(self):
        for sw_1 in self.pheromone.keys():
            for sw_2 in self.pheromone[sw_1].keys():
                self.pheromone[sw_1][sw_2]=self.pheromone[sw_1][sw_2]*(1-self.p)
        for i in range(self.N):
            for j in range(len(self.colony[i].path) - 1):
                p1 = self.colony[i].path[j]
                p2 = self.colony[i].path[j + 1]
                self.pheromone[p1][p2] += self.colony[i].delta*self.Q
                self.pheromone[p2][p1] += self.colony[i].delta*self.Q
     
    def MemorizeCondidates(self):
        self.colony.sort(key=lambda x: x.fitness)
        condidate = []
        k=0
        for i in range(len(self.colony)):
            dk_3 = False
            for ant in condidate:
                if(tuple(ant.path)==tuple(self.colony[i].path)):
                    dk_3 = True
                    break
            if(dk_3!=True):
                condidate.append(copy.deepcopy(self.colony[i]))
                k=k+1
            if(k==self.K_paths):
                break
        self.condidates.extend(copy.deepcopy(condidate))

    def GetBest(self):
        self.condidates.sort(key=lambda x: x.fitness)
        self.best.clear()
        k=0
        for i in range(len(self.condidates)):
            dk_3 = False
            for ant in self.best:
                if(tuple(ant.path)==tuple(self.condidates[i].path)):
                    dk_3 = True
                    break
            if(dk_3!=True):
                self.best.append(copy.deepcopy(self.condidates[i]))
                k=k+1
            if(k==self.K_paths):
                break
                
    def Do(self):
        for i in range(self.Max):
            self.CreatePath()
            self.UpdatePheromone()
            self.MemorizeCondidates()
        self.GetBest()

In [2]:
Times = 10
N = [15, 30, 60]
Max = 50
K_paths = 10
p = 0.2
a = 0.6
b = 0.4
p0 = 0.3
Q = 10

In [3]:
weight_map={}
temp = 0
with open('metric_data_60.txt') as f:
    for line in f:
        strt = line
        strt2 = strt.split(':')
        my_result = list(map(int, strt2[0].split(',')))
        if (temp!=my_result[0]):
            weight_map[my_result[0]]={}
        weight_map[my_result[0]][my_result[1]] = int(strt2[1])
        temp = my_result[0]
        
vertices = [i for i in range(1,61)]
alg_y = YenAlgorithm(weight_map,vertices,11,50,K_paths)
paths_vertices = alg_y.compute_shortest_paths()

In [ ]:
solan_1 = np.zeros(K_paths)
solan_2 = np.zeros(K_paths)
solan_3 = np.zeros(K_paths)
CD_1 = []
CD_2 = []
CD_3 = []
for i in range(Times):
    alg_1 = ACO(11,50,N[0], Max, K_paths, p, a, b, p0, Q)
    alg_1.Do()
    for j in range(len(alg_1.best)):
        if(tuple(alg_1.best[j].path)==tuple(paths_vertices[j])):
            solan_1[j]+=1
    s = 0
    for member in alg_1.best:
        for i in range(len(member.path)-1):
            s+= alg_1.weight_map[member.path[i]][member.path[i+1]]
    CD_1.append(s)
            
    alg_2 = ACO(11,50,N[1], Max, K_paths, p, a, b, p0, Q)
    alg_2.Do()
    for j in range(len(alg_2.best)):
        if(tuple(alg_2.best[j].path)==tuple(paths_vertices[j])):
            solan_2[j]+=1
    s = 0
    for member in alg_2.best:
        for i in range(len(member.path)-1):
            s+= alg_2.weight_map[member.path[i]][member.path[i+1]]
    CD_2.append(s)
    
    alg_3 = ACO(11,50,N[2], Max, K_paths, p, a, b, p0, Q)
    alg_3.Do()
    for j in range(len(alg_3.best)):
        if(tuple(alg_3.best[j].path)==tuple(paths_vertices[j])):
            solan_3[j]+=1
    s = 0
    for member in alg_3.best:
        for i in range(len(member.path)-1):
            s+= alg_3.weight_map[member.path[i]][member.path[i+1]]
    CD_3.append(s)

In [ ]:
print(solan_1)
print(solan_2)
print(solan_3)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
font = {'size'   : 11}
matplotlib.rc('font', **font)
# create data
x = np.arange(10)+1
width = 0.25
  
# plot data in grouped manner of bar type
fig = plt.figure(figsize=(10,8))
plt.bar(x-0.25, solan_1, width, color='red')
plt.bar(x, solan_2, width, color='green')
plt.bar(x+0.25, solan_3, width, color='blue')
plt.xlabel("Path")
plt.ylabel("Times")
plt.legend(["N = "+str(N[0]), "N = "+str(N[1]), "N = "+str(N[2])])
plt.grid()
plt.savefig("ACO_N_3.png",dpi=200)

In [ ]:
figg1 = plt.figure(figsize=(10,8))
avr_1 = int(sum(CD_1)/Times)
avr_2 = int(sum(CD_2)/Times)
avr_3 = int(sum(CD_3)/Times)
plt.plot(CD_1, label = "N = "+str(N[0])+ " (Average = "+str(avr_1)+")", linewidth = '3')
plt.plot(CD_2, label = "N = "+str(N[1])+ " (Average = "+str(avr_2)+")", linewidth = '3')
plt.plot(CD_3, label = "N = "+str(N[2])+ " (Average = "+str(avr_3)+")", linewidth = '3')
plt.legend(loc="upper right")
plt.xlabel("Time")
plt.ylabel("CD")
plt.grid()
plt.savefig("ACO_N_3_2.png",dpi=200)